# 01 – Données et analyse exploratoire

**Projet :** détection d'objets pour mesurer l'occupation des postes de travail (flex office) – comparaison **YOLO** vs **Faster R-CNN**.

**Ce notebook :**
1. télécharge un sous-ensemble de **COCO 2017** limité à 5 classes : `person`, `cell phone`, `laptop`, `chair`, `cup` ;
2. construit trois jeux : **train**, **val** et **test** ;
3. fait l'**analyse exploratoire** des annotations (combien d'objets, de quelle taille, combien par image) ;
4. exporte les données au format YOLO et les sauvegarde sur Google Drive pour les notebooks suivants.

> Pas besoin de GPU pour ce notebook : *Exécution → Modifier le type d'exécution → CPU* suffit.

## 1. Installation et connexion à Google Drive

**Pourquoi Google Drive ?** Une session Colab est effacée quand elle se ferme (au bout d'environ 90 minutes d'inactivité ou de 12 heures). Tout ce qu'on veut garder (données préparées, poids des modèles, résultats) doit donc être écrit sur Drive.

**FiftyOne** est une bibliothèque open source de gestion de jeux d'images. On l'utilise pour trois choses : télécharger **uniquement** les images COCO qui contiennent nos classes (le jeu complet fait plus de 25 Go), filtrer les annotations et exporter au format attendu par YOLO.

In [ ]:
# Pillow < 13 : évite un conflit avec Gradio, préinstallé sur Colab
!pip install -q fiftyone "pillow<13"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Paramètres du projet

Tout ce qui peut changer est regroupé ici. La **graine aléatoire** (`SEED`) fixe le hasard : relancer le notebook redonne exactement les mêmes images et le même découpage. C'est ce qu'on appelle la **reproductibilité**.

In [ ]:
import os

CLASSES = ["person", "cell phone", "laptop", "chair", "cup"]
SEED = 51
N_IMAGES_PAR_CLASSE = 700      # images d'entraînement tirées pour CHAQUE classe
PART_VALIDATION = 0.15         # 15 % des images d'entraînement servent à la validation

DOSSIER_DRIVE = "/content/drive/MyDrive/detection-objets"
DOSSIER_EXPORT = "/content/donnees"            # disque local de Colab (rapide)
DOSSIER_YOLO = f"{DOSSIER_EXPORT}/yolo"

os.makedirs(f"{DOSSIER_DRIVE}/donnees", exist_ok=True)
os.makedirs(f"{DOSSIER_DRIVE}/resultats", exist_ok=True)

## 3. Téléchargement du jeu d'entraînement (COCO *train2017*)

**Le problème du déséquilibre :** dans COCO, `person` apparaît dans environ 45 % des images, `cell phone` dans à peine 4 %. Si on tirait 3 500 images au hasard, le modèle verrait surtout des personnes et très peu de téléphones.

**La solution :** on tire **700 images pour chaque classe**, puis on fusionne. Une même image peut contenir plusieurs classes (une personne avec un ordinateur) et être tirée plusieurs fois. On ne la garde qu'**une fois, avec ses boîtes d'origine**, d'où un total inférieur à 5 × 700.

> 🐞 **Bug corrigé (v1 → v2)** : la première version fusionnait ces images au lieu de les ignorer, ce qui **dupliquait leurs boîtes** (35 % de boîtes en double dans le train). C'est l'analyse exploratoire qui l'a révélé : une image affichait 52 personnes, 52 chaises, 40 ordinateurs… tous des multiples de 4. Un contrôle automatique a été ajouté en section 6.

⏱️ Environ 10 minutes. Le premier appel télécharge aussi le fichier d'annotations COCO (environ 240 Mo).

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

trainval = fo.Dataset("bureau_trainval", overwrite=True)

for classe in CLASSES:
    morceau = foz.load_zoo_dataset(
        "coco-2017",
        split="train",
        label_types=["detections"],
        classes=[classe],
        max_samples=N_IMAGES_PAR_CLASSE,
        shuffle=True,
        seed=SEED,
        dataset_name=f"coco_train_{classe.replace(' ', '_')}",
        drop_existing_dataset=True,
    )
    # skip_existing=True : une image déjà présente (tirée pour une autre classe) n'est PAS refusionnée.
    # Sinon, FiftyOne ajouterait une seconde fois toutes ses boîtes -> annotations en double.
    trainval.merge_samples(morceau, key_field="filepath", skip_existing=True)
    print(f"{classe:<11} +{len(morceau)} images  ->  total : {len(trainval)}")

## 4. Téléchargement du jeu de test (COCO *val2017*)

On prend **toutes** les images de *val2017* contenant au moins une de nos classes, sans rééquilibrage : le test doit refléter la réalité.

**Le point clé pour l'entretien :** les poids pré-entraînés de YOLO et de Faster R-CNN ont été appris sur *train2017*. Ils n'ont **jamais vu** *val2017*. Notre test est donc honnête, et c'est **le même pour les deux modèles**, sinon la comparaison n'aurait pas de sens.

In [ ]:
test = foz.load_zoo_dataset(
    "coco-2017",
    split="validation",
    label_types=["detections"],
    classes=CLASSES,
    dataset_name="bureau_test",
    drop_existing_dataset=True,
)
print("Images de test :", len(test))

## 5. Nettoyage des annotations et découpage train / val

Deux filtres sur les **boîtes** (pas sur les images) :
- on ne garde que nos 5 classes : une image de bureau contient aussi des claviers, des livres… que le modèle doit apprendre à **ignorer** (ils deviennent du « fond ») ;
- on retire les boîtes **`iscrowd`** : dans COCO, une seule boîte entoure parfois toute une foule. Elle apprendrait au modèle à encadrer un groupe au lieu de chaque personne.

Puis on découpe les images d'entraînement en **85 % train / 15 % val** :
- **train** : le modèle apprend dessus ;
- **val** : on surveille l'entraînement (choix de la meilleure époque, détection du sur-apprentissage) ;
- **test** : utilisé **une seule fois**, à la fin, pour la mesure finale.

In [ ]:
from fiftyone import ViewField as F
import fiftyone.utils.random as four

filtre_boites = F("label").is_in(CLASSES) & (F("iscrowd") != 1)

four.random_split(trainval, {"train": 1 - PART_VALIDATION, "val": PART_VALIDATION}, seed=SEED)

trainval_propre = trainval.filter_labels("ground_truth", filtre_boites, only_matches=True)
vues = {
    "train": trainval_propre.match_tags("train"),
    "val":   trainval_propre.match_tags("val"),
    "test":  test.filter_labels("ground_truth", filtre_boites, only_matches=True),
}

for nom, vue in vues.items():
    print(f"{nom:<5} : {len(vue):>5} images, {vue.count('ground_truth.detections'):>6} objets")

## 6. Analyse exploratoire : des annotations à un tableau

On transforme chaque **boîte** en une **ligne** d'un tableau pandas. C'est le réflexe Data Analyst : une fois les données à plat, on peut tout compter, croiser et exporter vers Power BI.

**Taille des objets :** on reprend les seuils officiels de COCO, en surface de la boîte en pixels :
- **petit** : moins de 32 × 32 px ;
- **moyen** : de 32 × 32 à 96 × 96 px ;
- **grand** : au-delà.

COCO publie une mAP séparée pour chaque taille, car les petits objets sont nettement plus difficiles à détecter.

In [ ]:
import pandas as pd

trainval.compute_metadata()   # largeur et hauteur de chaque image
test.compute_metadata()

def taille_coco(aire_px):
    if aire_px < 32 ** 2:
        return "petit"
    if aire_px < 96 ** 2:
        return "moyen"
    return "grand"

lignes = []
for nom_split, vue in vues.items():
    for echantillon in vue.select_fields(["ground_truth", "metadata"]).iter_samples(progress=True):
        L, H = echantillon.metadata.width, echantillon.metadata.height
        image = os.path.basename(echantillon.filepath)
        for boite in echantillon.ground_truth.detections:
            x, y, w, h = boite.bounding_box        # coordonnées relatives (entre 0 et 1)
            larg_px, haut_px = w * L, h * H
            lignes.append({
                "split": nom_split,
                "image": image,
                "classe": boite.label,
                "largeur_px": round(larg_px, 1),
                "hauteur_px": round(haut_px, 1),
                "aire_relative": round(w * h, 5),  # part de l'image couverte par l'objet
                "taille": taille_coco(larg_px * haut_px),
            })

df = pd.DataFrame(lignes)

# Contrôle qualité : aucune boîte ne doit apparaître deux fois à l'identique dans une même image
cles = ["split", "image", "classe", "largeur_px", "hauteur_px", "aire_relative"]
doublons = df[df.split != "test"].duplicated(cles).sum()
print(f"Boîtes en double dans train/val : {doublons}")
assert doublons < 10, "Annotations dupliquées : vérifier la fusion des jeux (section 3)"

df.to_csv(f"{DOSSIER_DRIVE}/resultats/eda_annotations.csv", index=False)
print(df.shape)
df.head()

### 6.1 Combien d'objets par classe ?

In [ ]:
ORDRE_CLASSES = CLASSES
ORDRE_SPLITS = ["train", "val", "test"]

objets = pd.crosstab(df["classe"], df["split"]).reindex(index=ORDRE_CLASSES, columns=ORDRE_SPLITS)
images = (df.drop_duplicates(["split", "image", "classe"])
            .pipe(lambda d: pd.crosstab(d["classe"], d["split"]))
            .reindex(index=ORDRE_CLASSES, columns=ORDRE_SPLITS))

print("Nombre d'OBJETS (boîtes) par classe")
display(objets)
print("Nombre d'IMAGES contenant la classe")
display(images)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b4b2a9", "axes.labelcolor": "#3d3d3a",
    "xtick.color": "#5f5e5a", "ytick.color": "#3d3d3a",
    "axes.grid": True, "grid.color": "#e8e6dc", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "font.size": 11,
})
BLEU = "#2a78d6"

serie = objets["train"].sort_values()
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.barh(serie.index, serie.values, color=BLEU, height=0.6)
ax.grid(axis="y", visible=False)
for i, v in enumerate(serie.values):
    ax.text(v, i, f" {v:,}".replace(",", " "), va="center", color="#3d3d3a", fontsize=10)
ax.margins(x=0.1)
ax.set_title("Jeu d'entraînement : nombre d'objets annotés par classe", loc="left", fontsize=12)
ax.set_xlabel("Nombre d'objets")
plt.tight_layout()
plt.savefig(f"{DOSSIER_DRIVE}/resultats/eda_objets_par_classe.png", dpi=150)
plt.show()

**À observer :** malgré le tirage de 700 images par classe, `person` compte beaucoup plus d'**objets**, parce qu'une image contient souvent plusieurs personnes. Le déséquilibre est réduit, mais pas supprimé. Il faudra regarder la **mAP par classe**, et pas seulement la moyenne.

### 6.2 Quelle taille font les objets ?

In [ ]:
ORDRE_TAILLES = ["petit", "moyen", "grand"]
COULEURS_TAILLES = ["#86b6ef", "#2a78d6", "#104281"]   # du clair (petit) au foncé (grand)

parts = (pd.crosstab(df.loc[df.split == "train", "classe"],
                     df.loc[df.split == "train", "taille"], normalize="index")
           .reindex(index=ORDRE_CLASSES[::-1], columns=ORDRE_TAILLES) * 100)

fig, ax = plt.subplots(figsize=(8, 3.6))
gauche = pd.Series(0.0, index=parts.index)
for taille, couleur in zip(ORDRE_TAILLES, COULEURS_TAILLES):
    ax.barh(parts.index, parts[taille], left=gauche, color=couleur, height=0.6,
            label=taille, edgecolor="white", linewidth=2)
    gauche += parts[taille]
ax.set_xlim(0, 100)
ax.grid(axis="y", visible=False)
ax.set_xlabel("% des objets de la classe")
ax.set_title("Jeu d'entraînement : répartition des tailles d'objets (seuils COCO)", loc="left", fontsize=12, pad=30)
ax.legend(ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0), frameon=False)
plt.tight_layout()
plt.savefig(f"{DOSSIER_DRIVE}/resultats/eda_tailles_par_classe.png", dpi=150)
plt.show()

parts.round(1)

**À observer :** `cup` et `cell phone` sont majoritairement de **petits** objets. Hypothèse à vérifier dans le notebook de comparaison : ce seront les classes les moins bien détectées, et c'est sur les petits objets que YOLO et Faster R-CNN devraient le plus différer.

### 6.3 Combien d'objets par image ?

In [ ]:
par_image = df[df.split == "train"].groupby("image").size()

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(par_image.clip(upper=20), bins=range(1, 22), color=BLEU, rwidth=0.85)
ax.grid(axis="x", visible=False)
ax.set_xlabel("Nombre d'objets dans l'image (20 = 20 ou plus)")
ax.set_ylabel("Nombre d'images")
ax.set_title("Jeu d'entraînement : densité des scènes", loc="left", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Médiane : {par_image.median():.0f} objets par image – maximum : {par_image.max()}")

### 6.4 Regarder les données

Les chiffres ne suffisent pas : on vérifie **à l'œil** que les boîtes sont bien placées et bien nommées.

In [ ]:
import random
from PIL import Image
import matplotlib.patches as patches

COULEURS_CLASSES = dict(zip(CLASSES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))

random.seed(SEED)
exemples = random.sample(list(vues["train"].select_fields(["ground_truth", "metadata"])), 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, ech in zip(axes.flat, exemples):
    img = Image.open(ech.filepath)
    L, H = img.size
    ax.imshow(img)
    for b in ech.ground_truth.detections:
        x, y, w, h = b.bounding_box
        c = COULEURS_CLASSES[b.label]
        ax.add_patch(patches.Rectangle((x * L, y * H), w * L, h * H, fill=False, edgecolor=c, linewidth=2))
        ax.text(x * L, y * H - 4, b.label, color="white", fontsize=9,
                bbox=dict(facecolor=c, edgecolor="none", pad=1.5))
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Export au format YOLO et sauvegarde sur Drive

**Le format YOLO :** pour chaque image, un fichier `.txt` avec une ligne par objet :

```
<numéro_classe> <x_centre> <y_centre> <largeur> <hauteur>
```

Les coordonnées sont **relatives** (entre 0 et 1), donc indépendantes de la taille de l'image. Un fichier `dataset.yaml` indique où sont les images et le nom des classes.

On exporte **un seul format**. Dans le notebook Faster R-CNN, on écrira nous-mêmes la conversion vers le format attendu par torchvision : coordonnées en pixels `(x_min, y_min, x_max, y_max)` et classe 0 réservée au fond.

L'export se fait sur le disque local de Colab (rapide), puis **une seule archive** est copiée sur Drive. Copier des milliers de petits fichiers un par un vers Drive serait très lent.

In [ ]:
import shutil

shutil.rmtree(DOSSIER_YOLO, ignore_errors=True)
for nom_split, vue in vues.items():
    vue.export(
        export_dir=DOSSIER_YOLO,
        dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth",
        split=nom_split,
        classes=CLASSES,
    )

print(open(f"{DOSSIER_YOLO}/dataset.yaml").read())

In [ ]:
# Vérification : une étiquette au hasard
fichier = sorted(os.listdir(f"{DOSSIER_YOLO}/labels/train"))[0]
print(fichier)
print(open(f"{DOSSIER_YOLO}/labels/train/{fichier}").read())

In [ ]:
archive = shutil.make_archive(f"{DOSSIER_DRIVE}/donnees/bureau_coco5_yolo", "zip", DOSSIER_EXPORT, "yolo")
print(archive, f"{os.path.getsize(archive) / 1e6:.0f} Mo")

## Récapitulatif : ce qu'il faut savoir expliquer

| Question possible en entretien | Réponse courte |
|---|---|
| Pourquoi COCO et pas vos propres photos ? | C'est la référence du domaine, l'annotation est fiable et le protocole est reproductible par n'importe qui. |
| Pourquoi 5 classes ? | Elles correspondent au cas d'usage (occupation de postes) et restent entraînables sur un GPU gratuit. |
| Comment avez-vous géré le déséquilibre ? | Tirage de 700 images par classe. Il reste plus d'objets `person` que d'autres, donc j'analyse la mAP par classe. |
| Pourquoi trois jeux ? | Train pour apprendre, val pour régler, test touché une seule fois pour une mesure honnête. |
| Vos modèles étaient pré-entraînés sur COCO : n'est-ce pas tricher ? | C'est du *transfer learning*, la pratique standard. Le test se fait sur *val2017*, que les poids n'ont jamais vu. |
| Qu'avez-vous appris de l'exploration ? | `cup` et `cell phone` sont surtout de petits objets : j'anticipe qu'ils seront les moins bien détectés. |

**Fichiers produits sur Drive (`detection-objets/`) :**
- `donnees/bureau_coco5_yolo.zip` : les données prêtes pour l'entraînement ;
- `resultats/eda_annotations.csv` : une ligne par objet, réutilisable dans Power BI ;
- `resultats/eda_*.png` : les graphiques, pour le README.